# 📗 검색 결과로 답을 쓰게 하기 — RAG 완성하기

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

교안_01 에서 RAG 의 설계(질문 → 검색기 → 생성기)를 배우고, 교안_02 에서 ChromaDB 로 **검색기**를 만들었습니다. 지금 우리 손에는 질문을 던지면 관련 여행지 문서를 찾아 주는 도구가 있습니다. 이번 시간엔 그 위에 **생성기**를 얹습니다 — 찾아온 문서를 **OpenAI API** 로 넘겨, LLM 이 그 근거만 보고 답을 쓰게 만듭니다. 이걸로 **질문 → 검색 → 생성** 한 바퀴가 닫히고 비로소 RAG 가 완성됩니다.

## ⏪ 복습 — 지금까지 만든 것

- **RAG** 는 답하기 전에 관련 문서를 **검색**해 근거로 건네는 설계다(교안_01).
- 문서를 임베딩해 **ChromaDB 컬렉션**에 적재하고, `query(query_embeddings=..., n_results=K)` 로 **Top-K 검색**을 했다(교안_02).
- `where=` 로 **메타데이터 필터**를 함께 걸어 유형·지역·입장료로 후보를 좁혔다(교안_02).
- 14일차에서 **OpenAI API** 로 LLM 을 부르는 법(`chat.completions.create`, `system`·`user` 역할, `temperature`·`max_tokens`)을 배웠다. **오늘 그 둘을 잇는다.**

**오늘의 목표**

- [ ] 근거 없이 물었을 때와 근거를 붙여 물었을 때의 답이 **어떻게 다른지** 눈으로 확인한다.
- [ ] 검색 결과(Top-K 문서)를 프롬프트에 끼워 넣는 **컨텍스트 문자열**을 만든다.
- [ ] `system` 메시지로 **"자료에만 근거해 답하라"** 는 규칙을 건다.
- [ ] 검색과 생성을 한 함수 **`rag_answer(question)`** 로 묶는다.
- [ ] 자료에 없는 질문에는 **모른다고 답하게** 만들어 환각을 줄인다.
- [ ] 답과 함께 **출처 문서**를 보여 주어 사용자가 검증할 수 있게 한다.

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 준비 — 교안_02 의 검색기를 다시 세우기

이 노트북만 따로 열어도 돌아가도록, 교안_02 에서 만든 **검색기를 그대로 다시 세웁니다**. 여행지 데이터를 읽어 임베딩하고 `travel_guide` 컬렉션에 적재하는 것까지 **교안_02 와 똑같은 코드**라 여러분이 다시 타이핑할 필요는 없습니다 — 실행만 하세요.

In [ ]:
# [제공 코드] 교안_02 에서 만든 검색기를 이 노트북에 다시 세웁니다.
spots = pd.read_csv('data/travel_spots.csv')
doc_texts = spots['description'].tolist()
doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

# 오늘은 OpenAI 클라이언트도 쓰므로 이름을 chroma_client 로 구분한다
chroma_client = chromadb.EphemeralClient()
travel_col = chroma_client.get_or_create_collection(
    'travel_guide', metadata={'hnsw:space': 'cosine'})

travel_col.add(
    ids=spots['id'].tolist(),
    embeddings=doc_emb,
    documents=spots['description'].tolist(),
    metadatas=[{'name': n, 'region': r, 'type': t, 'entrance_fee': int(f)}
               for n, r, t, f in zip(spots['name'], spots['region'], spots['type'], spots['entrance_fee'])],
)

print('검색기 준비 완료 — 저장된 문서 수:', travel_col.count())

## 1. 왜 근거를 함께 줘야 하나 — 환각을 눈으로 보기

LLM 은 **학습한 것**을 바탕으로 그럴듯한 문장을 만듭니다. 그래서 **우리 데이터에만 있는 사실**을 물으면 곤란해집니다 — 모른다고 하기도 하고, 아는 척 지어내기도 하고, 우연히 맞히기도 합니다. 문제는 **받은 답이 맞는지 확인할 길이 없다**는 것입니다. 이렇게 근거 없이 그럴듯한 말을 지어내는 현상을 **환각(Hallucination)** 이라고 합니다.

해법은 단순합니다 — **답하기 전에 근거를 찾아서 같이 건네주는 것**입니다. 그것이 RAG 입니다.

먼저 LLM 을 부를 준비부터 합니다. 14일차와 **똑같은 방식**입니다 — 폴더의 `.env` 파일에 적어 둔 `OPENAI_API_KEY` 를 읽어 클라이언트를 만듭니다. **노트북에 키를 직접 적지 마세요.** 이 폴더의 `.env.example` 을 `.env` 로 복사한 뒤 본인 키를 채우면 됩니다.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 14일차에서 쓰던 그 방식 그대로입니다.
import os

from dotenv import load_dotenv   # .env 를 환경변수로 읽는 도구
from openai import OpenAI

load_dotenv('.env')                # 같은 폴더
load_dotenv('../.env')             # 정답 폴더용

# 키가 없으면 OpenAI() 에서 알아보기 어려운 에러가 나므로 먼저 확인한다
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(
        '이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n'
        '  1) 일차 폴더에서  cp .env.example .env\n'
        '  2) .env 를 열어 OPENAI_API_KEY 에 본인 키를 채우기\n'
        '  3) 커널을 다시 시작한 뒤 이 셀부터 실행'
    )

# max_retries : 분당 한도(429)에 걸렸을 때 자동으로 재시도할 횟수
client = OpenAI(max_retries=8)     # 키는 환경변수에서 자동으로 찾는다

print('연결 준비 완료 — 키 확인됨')

이제 **같은 질문을 두 번** 던져 비교합니다.

- **(가) 근거 없이 그냥 묻기** — 질문만 보낸다.
- **(나) 근거를 붙여 묻기** — 먼저 우리 컬렉션에서 검색해, 찾은 문서를 질문과 함께 보낸다.

질문은 "**소금산 출렁다리는 어디에 있고 입장료가 얼마야?**" 입니다. 우리 코퍼스에는 **강원 · 입장료 10000원**으로 적혀 있습니다. 이렇게 **우리 데이터 안에서만 확인 가능한 것**을 물어야 두 답의 차이가 드러납니다.

> 이 절에서는 근거 문서를 **이름으로 직접 집어 옵니다**(교안_02 에서 배운 `get(where=...)`). 여기서 보려는 것은 '근거를 붙이면 답이 어떻게 달라지는가' 하나뿐이라, 검색 단계를 끼워 넣지 않고 가장 단순하게 둔 것입니다. **질문으로 검색해서 가져오는 일은 §3 에서 함수로 묶습니다.**

In [ ]:
# ── (가) 근거 없이 그냥 묻기 ──
question = "소금산 출렁다리는 어디에 있고 입장료가 얼마야?"

plain = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    temperature=0,     # 답의 흔들림을 줄인다
    max_tokens=300,    # 길이·요금 상한
).choices[0].message.content

print(plain)

받은 답을 잠시 그대로 두고, **이번엔 같은 질문에 우리 자료를 붙여** 다시 물어봅시다. 달라지는 것이 무엇인지 곧 대조합니다.

In [ ]:
# ── (나) 근거를 붙여 묻기 ──
# 먼저 우리 컬렉션에서 그 문서를 꺼낸다(get(where=...) 는 교안_02 의 메타데이터 필터)
hit = travel_col.get(
    where={'name': '소금산 출렁다리'}, include=['documents', 'metadatas']
)
meta = hit['metadatas'][0]

fact = (
    f"{meta['name']} | 지역 {meta['region']} "
    f"| 입장료 {meta['entrance_fee']}원 | {hit['documents'][0]}"
)
print('[근거로 건넬 자료]', fact, '\n')

prompt = f'다음 자료만 보고 답하세요.\n\n[자료]\n{fact}\n\n[질문] {question}'

grounded = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}],
    temperature=0,
    max_tokens=300,
).choices[0].message.content

print(grounded)

**두 셀의 답을 나란히 놓고, 우리 데이터(강원 · 10000원)와 대조해 보세요.**

- **(나)** 는 우리가 건넨 자료를 읽고 쓴 답입니다. **지역과 입장료가 자료 그대로인지** 대조해 보세요. 틀렸다면 자료를 잘못 읽은 것이니 **어디서 틀렸는지 짚어 낼 수 있습니다**.
- **(가)** 는 사정이 다릅니다. 모른다고 물러설 수도, 그럴듯한 숫자를 지어낼 수도, 우연히 맞힐 수도 있습니다. **어느 쪽이든 우리는 그 답이 맞는지 확인할 방법이 없습니다** — 근거가 없기 때문입니다.

> ⚠️ **"근거 없이 물으면 반드시 틀린다"는 뜻이 아닙니다.** LLM 의 답은 실행할 때마다 조금씩 달라질 수 있고, 널리 알려진 곳이라면 맞힐 수도 있습니다. 요점은 **정확도**가 아니라 **검증 가능성**입니다 — RAG 는 답에 **출처를 붙여** 사람이 확인할 수 있게 만듭니다.

여기서 한 일이 사실 RAG 의 전부입니다. **검색해서 → 프롬프트에 끼워 넣고 → 물어본다.** 남은 절에서는 이 셋을 제대로 다듬습니다.

### 🖐️ 함께 따라하기

이번엔 **비교에 쓸 만한 질문거리**를 직접 찾아봅시다. LLM 이 알 리 없고 우리 코퍼스에만 있는 사실이라야 좋은 비교가 됩니다. **입장료가 비싼 곳**을 뽑아 보세요. (LLM 호출은 하지 않습니다 — 검색만 합니다.)

1. `travel_col.get(where={'entrance_fee': {'$gte': 3000}}, include=['metadatas'])` 로 입장료 3000원 이상인 문서를 모두 가져와 `pricey` 에 담는다.
2. `pricey['metadatas']` 를 돌며 각 문서의 `name`·`region`·`entrance_fee` 를 출력한다.
3. 그중 하나를 골라, (가)/(나) 비교에 쓸 질문을 한 문장 만들어 본다(주석으로 적어 두면 됩니다).

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. LLM 이 근거 없이 그럴듯한 말을 지어내는 현상을 무엇이라 부르나요?
2. 근거를 함께 건네면 좋은 점을 **정확도**가 아닌 다른 말로 표현하면 무엇인가요?

<details><summary>정답 보기</summary>

1. 환각(Hallucination). 2. **검증 가능성** — 답이 어떤 자료에서 나왔는지 사람이 확인할 수 있다.

</details>

## 2. 프롬프트에 근거를 끼워 넣는 법 — 컨텍스트 만들기

§1 에서는 문서 **한 건**을 손으로 이어 붙였습니다. 실제로는 Top-K(보통 3~5건)를 받으니, 여러 문서를 **하나의 문자열**로 잇는 일이 필요합니다. 이 문자열을 **컨텍스트(context)** 라고 부릅니다.

컨텍스트를 만들 때 지킬 두 가지가 있습니다.

1. **원문만 넣지 말고 메타데이터도 함께 넣습니다.** 우리 코퍼스의 `description` 에는 입장료가 적혀 있지 않습니다 — 입장료는 **메타데이터에만** 있습니다. 넣지 않으면 LLM 은 답할 재료가 없습니다.
2. **문서끼리 경계가 보이게 씁니다.** 줄 앞에 `-` 를 붙여 목록처럼 만들면 LLM 이 몇 건인지, 어디서 끊기는지 알아보기 쉽습니다.

그리고 **규칙**을 겁니다. 14일차에서 배운 `system` 역할이 이 자리입니다 — `user` 는 이번 질문이고, `system` 은 **대화 내내 지켜야 할 태도**입니다. 여기서는 "**자료에만 근거해 답하고, 자료에 없으면 모른다고 말하라**" 는 규칙을 줍니다. 이 한 문장이 §4 에서 환각을 막는 장치가 됩니다.

In [ ]:
def build_context(res):
    """검색 결과(res)를 LLM 에게 건넬 한 덩어리 문자열로 잇는다."""
    lines = []

    # [0] : 질문이 하나라 그 결과만 꺼낸다
    for doc, meta in zip(res['documents'][0], res['metadatas'][0]):
        lines.append(f"- {meta['name']} (지역 {meta['region']}, 유형 {meta['type']}, "
                     f"입장료 {meta['entrance_fee']}원): {doc}")

    return '\n'.join(lines)   # 문서 한 건이 한 줄


# 대문자 이름은 '한 번 정하고 바꾸지 않는 값'이라는 표시(관례)
SYSTEM_RULE = (
    '너는 여행 안내원이다. 반드시 [자료]에 적힌 내용만 근거로 답하라. '
    '자료에 없는 내용은 지어내지 말고 "자료에 없습니다"라고 답하라. 답은 세 문장 이내로 한다.'
)

정의는 여기까지입니다. 컨텍스트가 실제로 어떻게 생겼는지 눈으로 확인해 봅시다(아직 LLM 은 부르지 않습니다).

In [ ]:
demo_query = "바다에서 물놀이하기 좋은 곳"
demo_emb = emb_model.encode([demo_query], normalize_embeddings=True)
demo_res = travel_col.query(query_embeddings=demo_emb, n_results=3)

demo_context = build_context(demo_res)
print(demo_context)

세 건이 **한 줄씩** 이어진 문자열이 나옵니다. 이 덩어리를 질문과 함께 `user` 메시지에 넣으면 됩니다. 실제로 보낼 메시지가 어떤 모양인지도 한번 찍어 봅시다.

In [ ]:
# 실제로 보낼 messages 의 '모양만' 확인한다
messages = [
    {'role': 'system', 'content': SYSTEM_RULE},
    {'role': 'user', 'content': f'[자료]\n{demo_context}\n\n[질문] {demo_query}'},
]

print('[system]')
print(messages[0]['content'])
print('\n[user]')
print(messages[1]['content'])

`system` 에는 **규칙**만, `user` 에는 **자료 + 이번 질문**이 담깁니다. 자료와 질문 사이를 `[자료]`·`[질문]` 이라는 **표지**로 나눠 둔 것에 주목하세요 — 어디까지가 근거이고 어디부터가 질문인지 LLM 이 헷갈리지 않게 하는 간단하고 효과적인 방법입니다.

### 🖐️ 함께 따라하기

다른 질문으로 컨텍스트를 만들어 출력해 보세요. 질문은 "**단풍이 아름다운 산**", Top-K 는 **2** 입니다. (여기서도 LLM 호출은 없습니다.)

1. 질문을 `my_query` 에 담고 `normalize_embeddings=True` 로 임베딩한다.
2. `travel_col.query(query_embeddings=..., n_results=2)` 로 검색해 `my_res` 에 담는다.
3. `build_context(my_res)` 로 컨텍스트를 만들어 출력한다.
4. 몇 줄이 나왔는지, 입장료가 함께 들어갔는지 눈으로 확인한다.

In [ ]:
# 여기에 위 1~4 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. 검색한 문서들을 이어 붙여 프롬프트에 넣는 그 문자열을 무엇이라 부르나요?
2. "자료에 없으면 모른다고 답하라" 같은 **규칙**은 `system` 과 `user` 중 어느 역할에 넣나요?

<details><summary>정답 보기</summary>

1. 컨텍스트(context). 2. `system` — 대화 내내 지켜야 할 태도·규칙을 담는 자리다(`user` 는 이번 질문).

</details>

## 3. 검색 + 생성을 한 함수로 — `rag_answer(question)`

지금까지 한 일을 늘어놓으면 네 단계입니다.

```
질문 → ① 임베딩 → ② travel_col.query 로 Top-K 검색 → ③ build_context 로 컨텍스트 조립
     → ④ client.chat.completions.create 로 답 생성
```

이 넷을 **함수 하나**로 묶으면, 앞으로는 `rag_answer("...")` 한 줄로 RAG 를 쓸 수 있습니다. 이것이 우리가 만드는 **RAG 파이프라인**입니다.

모델은 `gpt-4o-mini`, `temperature=0`(같은 질문엔 되도록 같은 답), `max_tokens=400`(길이 상한)으로 고정합니다.

In [ ]:
def rag_answer(question, k=3):
    """질문을 받아 검색 → 컨텍스트 조립 → 생성까지 하고 답 문장을 돌려준다."""
    # ① 질문을 문서와 '같은 방식'으로 임베딩
    q_emb = emb_model.encode([question], normalize_embeddings=True)

    # ② Top-K 검색(k 기본값 3)
    res = travel_col.query(query_embeddings=q_emb, n_results=k)

    # ③ 검색 결과를 한 덩어리 컨텍스트로
    context = build_context(res)

    # ④ 규칙(system) + 자료·질문(user) 을 보내 답을 받는다
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': SYSTEM_RULE},
            {'role': 'user', 'content': f'[자료]\n{context}\n\n[질문] {question}'},
        ],
        temperature=0,
        max_tokens=400,
    )

    return resp.choices[0].message.content


# 서로 다른 질문 두 개로 돌려 봅니다(호출 2회).
for q in ["제주에서 조용히 쉴 만한 곳 추천해 줘", "입장료가 있는 고궁은 어디야?"]:
    print('Q.', q)
    print('A.', rag_answer(q))
    print('-' * 60)

답에 나온 여행지 이름을 `spots` 에서 찾아보세요. 학습된 지식이 아니라 **방금 검색해 건넨 자료**를 읽고 쓴 답이라면, 이름이 우리 코퍼스 안에 있어야 합니다. 이렇게 **답을 원자료와 대조할 수 있다**는 것이 §1 에서 말한 검증 가능성입니다.

> `k` 값은 **넉넉함과 잡음의 맞바꿈**입니다. 작게 잡으면 정작 필요한 문서가 빠질 수 있고, 크게 잡으면 관계없는 문서까지 들어가 답이 흐려지고 요금도 늘어납니다(자료가 길어지면 입력 토큰이 늘어납니다). 보통 3~5 에서 시작해 결과를 보며 조정합니다.

### 🖐️ 함께 따라하기

여러분의 질문으로 `rag_answer` 를 불러 보세요.

1. 물어보고 싶은 것을 `my_question` 에 담는다(예: "**아이와 가기 좋은 무료 관광지 알려 줘**").
2. `rag_answer(my_question, k=4)` 로 호출해 답을 출력한다(Top-4 를 근거로 쓰겠다는 뜻).
3. 답에 나온 여행지 이름이 `spots['name'].tolist()` 안에 실제로 있는지 확인해 본다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. `rag_answer` 안에서 일어나는 네 단계를 순서대로 말해 보세요.
2. `temperature=0` 으로 둔 이유는 무엇인가요?

<details><summary>정답 보기</summary>

1. 질문 임베딩 → Top-K 검색 → 컨텍스트 조립 → LLM 호출(생성). 2. 같은 질문에는 되도록 **같은 답**이 나오게 해, 실습·검증에서 결과가 덜 흔들리게 하려고.

</details>

## 4. 근거 없는 질문에는 뭐라고 답하나 — 모른다고 말하게 하기

검색기는 **언제나 K 건을 돌려줍니다.** 코퍼스에 관련 문서가 하나도 없어도, 그중 '그나마 가장 가까운' 문서를 K 건 골라 옵니다. 즉 **엉뚱한 자료가 컨텍스트로 들어가는 일이 생깁니다.**

이때 규칙이 없으면 LLM 은 그 엉뚱한 자료를 억지로 엮거나 아는 척 지어낼 수 있습니다. 우리가 `SYSTEM_RULE` 에 넣어 둔 "**자료에 없는 내용은 지어내지 말고 "자료에 없습니다"라고 답하라**" 가 바로 이 상황을 막는 장치입니다.

우리 코퍼스는 **국내 여행지**만 담고 있습니다. 여기 없는 것을 물어봅시다.

In [ ]:
outside_question = "파리 에펠탑 입장료 알려 줘"

# 관련이 없어도 검색기는 K 건을 준다
out_emb = emb_model.encode([outside_question], normalize_embeddings=True)
out_res = travel_col.query(query_embeddings=out_emb, n_results=3)

print('검색기가 가져온 문서:', [m['name'] for m in out_res['metadatas'][0]])

에펠탑과 상관없는 국내 여행지가 그대로 컨텍스트에 들어갑니다. **그 상태로** 규칙(`SYSTEM_RULE`)을 걸고 물으면 뭐라고 답하는지 봅시다.

In [ ]:
print('[규칙 있음]', rag_answer(outside_question))

나온 답이 **자료에 없다는 취지로 물러서는지** 확인해 보세요. 건네받은 엉뚱한 자료를 억지로 엮지 않고 물러서는 것, 그것이 **RAG 가 환각을 줄이는 핵심 장치**입니다.

> 순서를 다시 새겨 두세요 — **검색기가 걸러 주는 게 아닙니다.** 검색기는 늘 K 건을 주고, **"자료에 없으면 모른다고 하라"는 규칙이 마지막 문지기** 역할을 합니다. 그래서 RAG 를 만들 때 `system` 규칙 한 문장이 검색 품질만큼이나 중요합니다.

> ⚠️ 규칙을 걸었다고 **100% 보장되지는 않습니다.** LLM 은 확률적으로 답을 만들기 때문에 가끔 규칙을 벗어납니다. 그래서 실무에서는 규칙에 더해 **출처를 함께 보여 주고**(§5), **유사도가 너무 낮으면 아예 답하지 않는** 식의 장치를 겹쳐 겁니다.

### 🖐️ 함께 따라하기

**규칙을 빼면** 답이 어떻게 달라지는지 직접 비교해 보세요. 같은 질문을 `system` 없이 던집니다.

1. `out_context = build_context(out_res)` 로 위에서 검색한 결과의 컨텍스트를 만든다.
2. `client.chat.completions.create` 를 부르되 **`messages` 에 `system` 을 넣지 않고** `user` 하나만 넣는다(내용은 `f'[자료]\n{out_context}\n\n[질문] {outside_question}'`).
3. `model='gpt-4o-mini'`, `temperature=0`, `max_tokens=300` 으로 두고 답을 출력한다.
4. §4 의 "규칙 있음" 답과 나란히 놓고 어떻게 다른지 본다.

In [ ]:
# 여기에 위 1~4 단계를 직접 작성해 보세요.

> 규칙을 빼면 답이 **어느 쪽으로든 흔들릴 수 있습니다** — 자료와 무관한 자기 지식으로 에펠탑 입장료를 말하기도 하고, 건네받은 국내 여행지를 엉뚱하게 엮기도 합니다. 규칙을 걸었을 때처럼 **일관되게 물러서지는 않는다**는 점이 핵심입니다. 실행할 때마다 답이 조금씩 다를 수 있으니, 여러분이 받은 두 답을 직접 견줘 보세요.

### ✅ 바로 확인 퀴즈

1. 코퍼스에 관련 문서가 없을 때, 검색기는 아무것도 돌려주지 않나요?
2. 자료에 없는 질문에 LLM 이 지어내지 않도록 막는 장치는 무엇인가요?

<details><summary>정답 보기</summary>

1. 아니다 — **언제나 K 건**을 돌려준다(관련이 없어도 '그나마 가까운' 것을 준다). 2. `system` 규칙("자료에 없으면 모른다고 답하라"). 다만 100% 보장은 아니라 출처 제시 등을 함께 쓴다.

</details>

## 5. 출처를 함께 보여 주기

§1 에서 RAG 의 장점은 정확도가 아니라 **검증 가능성**이라고 했습니다. 그런데 답만 덜렁 보여 주면 사용자는 여전히 확인할 길이 없습니다. **어떤 문서를 근거로 삼았는지 함께 보여 줘야** 비로소 검증이 가능해집니다.

우리가 이미 갖고 있는 재료입니다 — `res['metadatas'][0]` 에 문서 이름이 들어 있으니, 답과 함께 돌려주기만 하면 됩니다. 함수가 값을 **두 개** 돌려주도록 고쳐 봅시다(파이썬은 `return a, b` 로 여러 값을 한 번에 돌려줄 수 있습니다).

In [ ]:
def rag_answer_with_sources(question, k=3):
    """답과 함께 '근거로 쓴 문서 이름 목록'을 돌려준다."""
    q_emb = emb_model.encode([question], normalize_embeddings=True)
    res = travel_col.query(query_embeddings=q_emb, n_results=k)
    context = build_context(res)

    # 컨텍스트에 넣은 바로 그 문서들의 이름
    sources = [m['name'] for m in res['metadatas'][0]]

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': SYSTEM_RULE},
            {'role': 'user', 'content': f'[자료]\n{context}\n\n[질문] {question}'},
        ],
        temperature=0,
        max_tokens=400,
    )

    return resp.choices[0].message.content, sources   # 값 두 개를 반환


src_question = "역사 유적을 둘러보고 싶어. 어디가 좋을까?"

# 두 값을 각각 변수로 받는다(언패킹)
answer, sources = rag_answer_with_sources(src_question)

print('Q.', src_question)
print('A.', answer)
print('\n📚 근거로 쓴 문서:', ', '.join(sources))

답 아래에 **근거 문서 목록**이 함께 나옵니다. 사용자는 그 이름을 보고 "이 답이 어디서 나왔는지" 바로 확인할 수 있고, 이상하면 원문을 찾아볼 수 있습니다.

> 주의 — 여기 나오는 출처는 **컨텍스트로 건넨 문서 전부**이지, LLM 이 실제로 답에 쓴 문서만은 아닙니다. K 건을 줬으니 K 건이 그대로 찍힙니다. 답에 쓴 문서만 정확히 표시하려면 LLM 에게 **"어떤 자료를 썼는지 함께 적으라"** 고 시켜야 하는데, 그건 16일차 이후 다룹니다.

### 🖐️ 함께 따라하기

출처가 있으면 **사용자가 직접 검증할 수 있다**고 했습니다. 방금 받은 `sources` 로 그 검증을 실제로 해 봅시다. (LLM 을 다시 부르지 않습니다 — 위에서 받은 결과를 그대로 씁니다.)

1. `sources` 를 **번호를 붙여** 한 줄씩 출력한다 (`for i, name in enumerate(sources, 1):` 을 쓰면 1부터 번호가 붙습니다).
2. 같은 반복 안에서, 그 이름의 **원문**을 `spots` 에서 찾아 함께 출력한다 — `row = spots[spots['name'] == name].iloc[0]` 로 그 행을 꺼내 `row['description']` 을 쓴다.
3. 위 답이 이 원문들에서 나온 말인지 눈으로 대조한다. **이것이 출처를 붙이는 이유**입니다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. 답과 함께 출처를 보여 주면 사용자에게 어떤 점이 좋아지나요?
2. 파이썬 함수가 값을 두 개 돌려주게 하려면 어떻게 쓰나요?

<details><summary>정답 보기</summary>

1. 답이 **어떤 자료에서 나왔는지 직접 확인**할 수 있어 신뢰가 생긴다(검증 가능성). 2. `return 답, 출처` 처럼 쉼표로 나열하고, 받을 때는 `답, 출처 = 함수(...)` 로 언패킹한다.

</details>

## 이번 강의 정리

- **검색기(교안_02) + 생성기(오늘) = RAG.** 질문 → 임베딩 → Top-K 검색 → 컨텍스트 조립 → LLM 생성.
- 근거 없이 물으면 답이 맞는지 **확인할 길이 없다**. RAG 의 이점은 정확도 이전에 **검증 가능성**이다.
- **컨텍스트**는 검색한 문서들을 한 문자열로 이은 것. 원문뿐 아니라 **메타데이터(입장료 등)** 도 넣어야 LLM 이 답할 재료를 갖는다.
- **`system` 규칙**으로 "자료에만 근거해 답하고, 없으면 모른다고 하라"를 건다 — 환각을 줄이는 핵심 장치.
- 검색기는 관련 문서가 없어도 **언제나 K 건**을 돌려준다. 그래서 마지막 문지기는 규칙이다.
- 답과 함께 **출처**를 보여 주어야 사용자가 검증할 수 있다.
- 호출 파라미터: `model='gpt-4o-mini'`, `temperature=0`(흔들림 줄이기), `max_tokens`(길이·요금 상한).

## ⏭️ 예고 — 16일차: RAG 파이프라인 다듬기

오늘 만든 RAG 는 **한 문서가 짧아서** 그대로 넣을 수 있었습니다. 실제 문서는 수십 쪽짜리라 그대로는 들어가지 않습니다. 16일차에서는 긴 문서를 알맞은 크기로 자르는 **청킹(chunking)** 과, 만든 RAG 가 얼마나 잘 답하는지 재는 **품질 지표**를 배웁니다.